In [1]:
!pip install nba_api tqdm
import pandas as pd
import numpy as np
from nba_api.stats.endpoints import leaguegamelog
from tqdm import tqdm
import time


In [2]:
#pulling the seasons 2024-2000 from the api
headers = {
    'Host': 'stats.nba.com',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Referer': 'https://www.nba.com/',
    'Origin': 'https://www.nba.com'
}


seasons = []

for year in range(2000, 2025):
    seasons.append(f"{year}-{str(year+1)[-2:]}")

all_games = []

for season in tqdm(seasons):

    try:
        gamelog = leaguegamelog.LeagueGameLog(
            season=season,
            season_type_all_star='Regular Season',
            headers=headers
        )

        df = gamelog.get_data_frames()[0]
        df["SEASON"] = season

        all_games.append(df)

        time.sleep(2)

    except Exception as e:
        print("Failed season:", season)
        print(e)
        time.sleep(5)



100%|██████████| 25/25 [00:54<00:00,  2.16s/it]


In [3]:
print(df.shape)


(2460, 30)


In [4]:
df.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,SEASON
0,22024,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,240,48,...,40,33,6,3,4,15,132,23,1,2024-25
1,22024,1610612750,MIN,Minnesota Timberwolves,0022400062,2024-10-22,MIN @ LAL,L,240,35,...,47,17,4,1,16,22,103,-7,1,2024-25
2,22024,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,240,42,...,46,22,7,8,7,22,110,7,1,2024-25
3,22024,1610612752,NYK,New York Knicks,0022400061,2024-10-22,NYK @ BOS,L,240,43,...,34,20,2,3,12,12,109,-23,1,2024-25
4,22024,1610612744,GSW,Golden State Warriors,0022400072,2024-10-23,GSW @ POR,W,240,48,...,57,38,13,5,18,27,140,36,1,2024-25


In [5]:
#shape of the data
games_df = pd.concat(all_games, ignore_index=True)

print(games_df.shape)


(60050, 30)


In [6]:
#total number of games in each of the seasons in the dataset
games_df["SEASON"].value_counts().sort_index()


SEASON
2000-01    2378
2001-02    2378
2002-03    2378
2003-04    2378
2004-05    2460
2005-06    2460
2006-07    2460
2007-08    2460
2008-09    2460
2009-10    2460
2010-11    2460
2011-12    1980
2012-13    2460
2013-14    2460
2014-15    2460
2015-16    2460
2016-17    2460
2017-18    2460
2018-19    2460
2019-20    2118
2020-21    2160
2021-22    2460
2022-23    2460
2023-24    2460
2024-25    2460
Name: count, dtype: int64

In [7]:
#merge home and away team stats for a game inot one single line of data 
#otherwise games will be counted twice 

games_df["HOME"] = games_df["MATCHUP"].str.contains("vs").astype(int)
home_df = games_df[games_df["HOME"] == 1]
away_df = games_df[games_df["HOME"] == 0]
merged_df = home_df.merge(
    away_df,
    on="GAME_ID",
    suffixes=("_HOME", "_AWAY")
)
print(merged_df.shape)


(30020, 61)


In [8]:
#make sure the data is ready for modeling
merged_df.duplicated(subset="GAME_ID").sum()

merged_df["HOME_WIN"] = (merged_df["WL_HOME"] == "W").astype(int)

merged_df.describe()


,TEAM_ID_HOME,MIN_HOME,FGM_HOME,FGA_HOME,FG_PCT_HOME,FG3M_HOME,FG3A_HOME,FG3_PCT_HOME,FTM_HOME,FTA_HOME,...,AST_AWAY,STL_AWAY,BLK_AWAY,TOV_AWAY,PF_AWAY,PTS_AWAY,PLUS_MINUS_AWAY,VIDEO_AVAILABLE_AWAY,HOME_AWAY,HOME_WIN
count,3.002000e+04,30020.000000,30020.000000,30020.000000,30019.000000,30020.000000,30020.000000,30019.000000,30020.000000,30020.000000,...,30020.000000,30020.000000,30020.000000,30020.000000,30020.000000,30020.000000,30020.000000,30020.000000,30020.0,30020.000000
mean,1.610613e+09,241.731679,38.680280,83.760160,0.462651,8.449667,23.368854,0.358424,18.362858,24.072818,...,22.029714,7.579614,4.647935,14.488008,20.947801,101.417988,-2.755097,0.524184,0.0,0.585943
std,8.623176e+00,7.504740,5.545734,8.049783,0.056638,4.450568,10.145713,0.112244,6.256806,7.719660,...,5.307932,2.899661,2.451434,3.984391,4.560724,13.942128,13.586447,0.508937,0.0,0.492567
min,1.610613e+09,0.000000,0.000000,0.000000,0.247000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-73.000000,0.000000,0.0,0.000000
25%,1.610613e+09,240.000000,35.000000,78.000000,0.424000,5.000000,15.000000,0.286000,14.000000,19.000000,...,18.000000,6.000000,3.000000,12.000000,18.000000,92.000000,-11.000000,0.000000,0.0,0.000000
50%,1.610613e+09,240.000000,39.000000,83.000000,0.462000,8.000000,22.000000,0.357000,18.000000,24.000000,...,22.000000,7.000000,4.000000,14.000000,21.000000,101.000000,-4.000000,1.000000,0.0,1.000000
75%,1.610613e+09,240.000000,42.000000,89.000000,0.500000,11.000000,31.000000,0.429000,22.000000,29.000000,...,25.000000,9.000000,6.000000,17.000000,24.000000,111.000000,7.000000,1.000000,0.0,1.000000
max,1.610613e+09,340.000000,65.000000,125.000000,0.689000,29.000000,70.000000,1.000000,48.000000,64.000000,...,48.000000,22.000000,19.000000,33.000000,42.000000,176.000000,57.000000,2.000000,0.0,1.000000


In [9]:
merged_df["HOME_WIN"].value_counts()


HOME_WIN
1    17590
0    12430
Name: count, dtype: int64

In [10]:
merged_df.isna().sum()

SEASON_ID_HOME            0
TEAM_ID_HOME              0
TEAM_ABBREVIATION_HOME    0
TEAM_NAME_HOME            0
GAME_ID                   0
                         ..
PLUS_MINUS_AWAY           0
VIDEO_AVAILABLE_AWAY      0
SEASON_AWAY               0
HOME_AWAY                 0
HOME_WIN                  0
Length: 62, dtype: int64

In [11]:
merged_df = merged_df.drop_duplicates()


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report

In [13]:
merged_df = merged_df.sort_values("GAME_DATE_HOME")




In [14]:
# Home team rolling stats
merged_df["HOME_PTS_LAST5"] = (
    merged_df.groupby("TEAM_ID_HOME")["PTS_HOME"]
    .transform(lambda x: x.shift(1).rolling(10).mean())
)

merged_df["HOME_REB_LAST5"] = (
    merged_df.groupby("TEAM_ID_HOME")["REB_HOME"]
    .transform(lambda x: x.shift(1).rolling(10).mean())
)

merged_df["HOME_AST_LAST5"] = (
    merged_df.groupby("TEAM_ID_HOME")["AST_HOME"]
    .transform(lambda x: x.shift(1).rolling(10).mean())
)

merged_df["HOME_TOV_LAST5"] = (
    merged_df.groupby("TEAM_ID_HOME")["TOV_HOME"]
    .transform(lambda x: x.shift(1).rolling(10).mean())
)


# Away team rolling stats
merged_df["AWAY_PTS_LAST5"] = (
    merged_df.groupby("TEAM_ID_AWAY")["PTS_AWAY"]
    .transform(lambda x: x.shift(1).rolling(10).mean())
)

merged_df["AWAY_REB_LAST5"] = (
    merged_df.groupby("TEAM_ID_AWAY")["REB_AWAY"]
    .transform(lambda x: x.shift(1).rolling(10).mean())
)

merged_df["AWAY_AST_LAST5"] = (
    merged_df.groupby("TEAM_ID_AWAY")["AST_AWAY"]
    .transform(lambda x: x.shift(1).rolling(10).mean())
)

merged_df["AWAY_TOV_LAST5"] = (
    merged_df.groupby("TEAM_ID_AWAY")["TOV_AWAY"]
    .transform(lambda x: x.shift(1).rolling(10).mean())
)

merged_df['HOME_PTS_DIFF_LAST5'] = (
    merged_df.groupby('TEAM_ID_HOME')['PTS_HOME']
    .shift(1).rolling(10).apply(lambda x: (x - merged_df.loc[x.index, 'PTS_AWAY']).mean())
    .reset_index(level=0, drop=True)
)

# Away team point differential over last 5 games
merged_df['AWAY_PTS_DIFF_LAST5'] = (
    merged_df.groupby('TEAM_ID_AWAY')['PTS_AWAY']
    .shift(1).rolling(10).apply(lambda x: (x - merged_df.loc[x.index, 'PTS_HOME']).mean())
    .reset_index(level=0, drop=True)
)



In [15]:
merged_df['GAME_DATE_HOME'] = pd.to_datetime(merged_df['GAME_DATE_HOME'])
merged_df['GAME_DATE_AWAY'] = pd.to_datetime(merged_df['GAME_DATE_AWAY'])

merged_df = merged_df.sort_values(by='GAME_DATE_HOME')

merged_df['HOME_REST_DAYS'] = merged_df.groupby('TEAM_ID_HOME')['GAME_DATE_HOME'].diff().dt.days
merged_df['AWAY_REST_DAYS'] = merged_df.groupby('TEAM_ID_AWAY')['GAME_DATE_AWAY'].diff().dt.days

merged_df['REST_DIFF'] = merged_df['HOME_REST_DAYS'] - merged_df['AWAY_REST_DAYS']

merged_df['HOME_WIN_LAST5'] = merged_df.groupby('TEAM_ID_HOME')['HOME_WIN']\
    .transform(lambda x: x.rolling(10).mean().shift(1))

merged_df['AWAY_WIN_LAST5'] = merged_df.groupby('TEAM_ID_AWAY')['HOME_WIN']\
    .transform(lambda x: x.rolling(10).mean().shift(1))

merged_df['WIN_PCT_DIFF'] = merged_df['HOME_WIN_LAST5'] - merged_df['AWAY_WIN_LAST5']


In [16]:
merged_df['HOME_FG_PCT_LAST5'] = merged_df.groupby('TEAM_ID_HOME')['FG_PCT_HOME']\
    .transform(lambda x: x.shift(1).rolling(10).mean())

merged_df['AWAY_FG_PCT_LAST5'] = merged_df.groupby('TEAM_ID_AWAY')['FG_PCT_AWAY']\
    .transform(lambda x: x.shift(1).rolling(10).mean())

merged_df['FG_PCT_LAST5_DIFF'] = (
    merged_df['HOME_FG_PCT_LAST5'] - merged_df['AWAY_FG_PCT_LAST5']
)


In [17]:
## Creating ELO Variable 
# Set baseline ELO
BASE_ELO = 1500
K = 23  # adjustment factor
team_elo = {}

# Fill initial ELO
teams = pd.concat([merged_df['TEAM_ID_HOME'], merged_df['TEAM_ID_AWAY']]).unique()
for t in teams:
    team_elo[t] = BASE_ELO
    

    
    
    
home_elo_list = []
away_elo_list = []

for idx, row in merged_df.iterrows():
    home_id = row['TEAM_ID_HOME']
    away_id = row['TEAM_ID_AWAY']
    
    home_elo = team_elo[home_id]
    away_elo = team_elo[away_id]
    
    # Save current ELOs
    home_elo_list.append(home_elo)
    away_elo_list.append(away_elo)
    
    # Expected probability of home win
    expected_home = 1 / (1 + 10 ** ((away_elo - home_elo) / 400))
    
    # Actual result (1 if home wins, 0 if loses)
    actual_home = 1 if row['HOME_WIN'] == 1 else 0
    
    # Update ratings
    team_elo[home_id] += K * (actual_home - expected_home)
    team_elo[away_id] += K * ((1 - actual_home) - (1 - expected_home))  # same as -K*(actual_home - expected_home)
merged_df['HOME_ELO'] = home_elo_list
merged_df['AWAY_ELO'] = away_elo_list
merged_df['ELO_DIFF'] = merged_df['HOME_ELO'] - merged_df['AWAY_ELO']


In [18]:
merged_df.columns

Index(['SEASON_ID_HOME', 'TEAM_ID_HOME', 'TEAM_ABBREVIATION_HOME',
       'TEAM_NAME_HOME', 'GAME_ID', 'GAME_DATE_HOME', 'MATCHUP_HOME',
       'WL_HOME', 'MIN_HOME', 'FGM_HOME', 'FGA_HOME', 'FG_PCT_HOME',
       'FG3M_HOME', 'FG3A_HOME', 'FG3_PCT_HOME', 'FTM_HOME', 'FTA_HOME',
       'FT_PCT_HOME', 'OREB_HOME', 'DREB_HOME', 'REB_HOME', 'AST_HOME',
       'STL_HOME', 'BLK_HOME', 'TOV_HOME', 'PF_HOME', 'PTS_HOME',
       'PLUS_MINUS_HOME', 'VIDEO_AVAILABLE_HOME', 'SEASON_HOME', 'HOME_HOME',
       'SEASON_ID_AWAY', 'TEAM_ID_AWAY', 'TEAM_ABBREVIATION_AWAY',
       'TEAM_NAME_AWAY', 'GAME_DATE_AWAY', 'MATCHUP_AWAY', 'WL_AWAY',
       'MIN_AWAY', 'FGM_AWAY', 'FGA_AWAY', 'FG_PCT_AWAY', 'FG3M_AWAY',
       'FG3A_AWAY', 'FG3_PCT_AWAY', 'FTM_AWAY', 'FTA_AWAY', 'FT_PCT_AWAY',
       'OREB_AWAY', 'DREB_AWAY', 'REB_AWAY', 'AST_AWAY', 'STL_AWAY',
       'BLK_AWAY', 'TOV_AWAY', 'PF_AWAY', 'PTS_AWAY', 'PLUS_MINUS_AWAY',
       'VIDEO_AVAILABLE_AWAY', 'SEASON_AWAY', 'HOME_AWAY', 'HOME_WIN',
       'H

In [19]:

merged_df = merged_df.dropna(subset=[
    'HOME_PTS_LAST5', 'AWAY_PTS_LAST5',
    'HOME_REB_LAST5', 'AWAY_REB_LAST5', 'HOME_AST_LAST5', 
    'AWAY_AST_LAST5','HOME_TOV_LAST5', 'AWAY_TOV_LAST5',
    'HOME_PTS_DIFF_LAST5', 'AWAY_PTS_DIFF_LAST5', 'HOME_REST_DAYS',
    'AWAY_REST_DAYS', 'REST_DIFF','WIN_PCT_DIFF',
    'HOME_FG_PCT_LAST5','AWAY_FG_PCT_LAST5','FG_PCT_LAST5_DIFF','HOME_ELO', 'AWAY_ELO',
       'ELO_DIFF'
])
print("done")

done


In [20]:
# Extract starting year as integer
merged_df['SEASON_START'] = merged_df['SEASON_HOME'].str[:4].astype(int)

# Split dataset
train_df = merged_df[merged_df['SEASON_START'] <= 2019]
test_df  = merged_df[merged_df['SEASON_START'] >= 2020]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (23659, 85)
Test shape: (5995, 85)


In [21]:
#basic logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report, confusion_matrix

# Predictor variables
predictors = [
   'HOME_PTS_LAST5', 'AWAY_PTS_LAST5',
    'HOME_REB_LAST5', 'AWAY_REB_LAST5', 'HOME_AST_LAST5', 
    'AWAY_AST_LAST5','HOME_TOV_LAST5', 'AWAY_TOV_LAST5',
    'HOME_PTS_DIFF_LAST5', 'AWAY_PTS_DIFF_LAST5', 'HOME_REST_DAYS',
    'AWAY_REST_DAYS', 'REST_DIFF','WIN_PCT_DIFF',
    'HOME_FG_PCT_LAST5','AWAY_FG_PCT_LAST5','FG_PCT_LAST5_DIFF','HOME_ELO', 'AWAY_ELO',
       'ELO_DIFF'
]

X_train = train_df[predictors]
y_train = train_df['HOME_WIN']

X_test = test_df[predictors]
y_test = test_df['HOME_WIN']

# Train logistic regression
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

# Predict and evaluate
y_pred = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Log Loss:", log_loss(y_test, y_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.6412010008340283
Log Loss: 0.6301485315259325

Classification Report:
               precision    recall  f1-score   support

           0       0.64      0.45      0.53      2690
           1       0.64      0.80      0.71      3305

    accuracy                           0.64      5995
   macro avg       0.64      0.62      0.62      5995
weighted avg       0.64      0.64      0.63      5995


Confusion Matrix:
 [[1214 1476]
 [ 675 2630]]


C:\Users\jorda\anaconda3\New folder\envs\DS380\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [22]:
#Regularized Logistic Regression

logreg_l2 = LogisticRegression(
    penalty='l2',
    C=0.1,
    max_iter=1000
)

logreg_l2.fit(X_train, y_train)

y_pred_lr = logreg_l2.predict(X_test)
y_prob_lr = logreg_l2.predict_proba(X_test)[:,1]

# Get the coefficients
coefs = logreg_l2.coef_[0]  # [0] because coef_ is 2D: shape (1, n_features)

# Create a DataFrame to match coefficients with feature names
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': coefs
})

# Scale by 1000 for readability and round
coef_df['Coefficient'] = (coef_df['Coefficient'] * 1000).round(2)

# Optional: sort by absolute value for plotting
coef_df['abs_coef'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='abs_coef', ascending=True)

# Drop the helper column
coef_df = coef_df.drop(columns='abs_coef')

print(coef_df)
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

                Feature  Coefficient
10       HOME_REST_DAYS         0.03
16    FG_PCT_LAST5_DIFF        -0.08
12            REST_DIFF        -0.08
11       AWAY_REST_DAYS         0.11
18             AWAY_ELO        -1.48
17             HOME_ELO         1.63
0        HOME_PTS_LAST5         2.80
19             ELO_DIFF         3.11
5        AWAY_AST_LAST5        -3.14
3        AWAY_REB_LAST5        -5.18
1        AWAY_PTS_LAST5        -5.51
2        HOME_REB_LAST5         6.15
14    HOME_FG_PCT_LAST5         6.27
15    AWAY_FG_PCT_LAST5         6.35
7        AWAY_TOV_LAST5         9.38
6        HOME_TOV_LAST5        -9.55
4        HOME_AST_LAST5        15.75
8   HOME_PTS_DIFF_LAST5        22.34
9   AWAY_PTS_DIFF_LAST5       -29.35
13         WIN_PCT_DIFF        55.51
Accuracy: 0.6403669724770642
Confusion Matrix:
 [[1210 1480]
 [ 676 2629]]


C:\Users\jorda\anaconda3\New folder\envs\DS380\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [23]:
#gradient boosting model

from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3
)

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)
y_prob_gb = gb.predict_proba(X_test)[:,1]


print("=== Gradient Boosting ===")


# 2. Feature importance
print("Feature Importances:")
for f, imp in zip(X_train.columns, gb.feature_importances_):
    print(f, ":", round(imp, 4))

# 3. Performance
print("Accuracy:", accuracy_score(y_test, y_pred_gb))
print("Log Loss:", log_loss(y_test, y_prob_gb))

print("\nClassification Report:\n", classification_report(y_test, y_pred_gb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_gb))
print("\n")



=== Gradient Boosting ===
Feature Importances:
HOME_PTS_LAST5 : 0.0091
AWAY_PTS_LAST5 : 0.0095
HOME_REB_LAST5 : 0.0067
AWAY_REB_LAST5 : 0.0083
HOME_AST_LAST5 : 0.0138
AWAY_AST_LAST5 : 0.0076
HOME_TOV_LAST5 : 0.0069
AWAY_TOV_LAST5 : 0.0066
HOME_PTS_DIFF_LAST5 : 0.0271
AWAY_PTS_DIFF_LAST5 : 0.0434
HOME_REST_DAYS : 0.0037
AWAY_REST_DAYS : 0.0065
REST_DIFF : 0.0031
WIN_PCT_DIFF : 0.003
HOME_FG_PCT_LAST5 : 0.0122
AWAY_FG_PCT_LAST5 : 0.0088
FG_PCT_LAST5_DIFF : 0.0055
HOME_ELO : 0.0201
AWAY_ELO : 0.0162
ELO_DIFF : 0.7819
Accuracy: 0.6475396163469558
Log Loss: 0.6321796517100964

Classification Report:
               precision    recall  f1-score   support

           0       0.64      0.48      0.55      2690
           1       0.65      0.79      0.71      3305

    accuracy                           0.65      5995
   macro avg       0.65      0.63      0.63      5995
weighted avg       0.65      0.65      0.64      5995

Confusion Matrix:
 [[1284 1406]
 [ 707 2598]]




In [24]:
#Random Forests

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]


print("=== Random Forest ===")

# 2. Feature importance
print("Feature Importances:")
for f, imp in zip(X_train.columns, rf.feature_importances_):
    print(f, ":", round(imp, 4))

# 3. Performance
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Log Loss:", log_loss(y_test, y_prob_rf))

print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\n")


=== Random Forest ===
Feature Importances:
HOME_PTS_LAST5 : 0.0326
AWAY_PTS_LAST5 : 0.0306
HOME_REB_LAST5 : 0.029
AWAY_REB_LAST5 : 0.0299
HOME_AST_LAST5 : 0.0324
AWAY_AST_LAST5 : 0.0311
HOME_TOV_LAST5 : 0.0292
AWAY_TOV_LAST5 : 0.031
HOME_PTS_DIFF_LAST5 : 0.0375
AWAY_PTS_DIFF_LAST5 : 0.0432
HOME_REST_DAYS : 0.0151
AWAY_REST_DAYS : 0.015
REST_DIFF : 0.0192
WIN_PCT_DIFF : 0.0183
HOME_FG_PCT_LAST5 : 0.0382
AWAY_FG_PCT_LAST5 : 0.0356
FG_PCT_LAST5_DIFF : 0.0481
HOME_ELO : 0.13
AWAY_ELO : 0.098
ELO_DIFF : 0.256
Accuracy: 0.640533778148457
Log Loss: 0.6354561957306241

Classification Report:
               precision    recall  f1-score   support

           0       0.64      0.45      0.53      2690
           1       0.64      0.79      0.71      3305

    accuracy                           0.64      5995
   macro avg       0.64      0.62      0.62      5995
weighted avg       0.64      0.64      0.63      5995

Confusion Matrix:
 [[1216 1474]
 [ 681 2624]]




## Slight Change
I was getting the same results from each of the complex model as the simple linear regression. I found out that the reason was because of the predictor variables i was using and how they were not good enough. 

### I added a point diff variable and am still getting just above a 56% accuracy score with each of the variables

In [25]:

X = merged_df[predictors]
y = merged_df['HOME_WIN']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


#validation split
X_train_base, X_val, y_train_base, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

## train the base models
lr = LogisticRegression(max_iter=20000)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)

lr.fit(X_train_base, y_train_base)
rf.fit(X_train_base, y_train_base)
gb.fit(X_train_base, y_train_base)

#creats stacked features
lr_val = lr.predict_proba(X_val)[:, 1]
rf_val = rf.predict_proba(X_val)[:, 1]
gb_val = gb.predict_proba(X_val)[:, 1]

X_meta = np.column_stack((lr_val, rf_val, gb_val))

#train stacked model on features
meta_model = LogisticRegression()
meta_model.fit(X_meta, y_val)

#results section
lr_test = lr.predict_proba(X_test)[:, 1]
rf_test = rf.predict_proba(X_test)[:, 1]
gb_test = gb.predict_proba(X_test)[:, 1]

X_meta_test = np.column_stack((lr_test, rf_test, gb_test))

final_preds = meta_model.predict(X_meta_test)
final_probs = meta_model.predict_proba(X_meta_test)[:, 1]

#print results
print("Accuracy:", accuracy_score(y_test, final_preds))
print("Log Loss:", log_loss(y_test, final_probs))
print("Meta Model Weights:", meta_model.coef_)



Accuracy: 0.6624515258809645
Log Loss: 0.6138687261845353
Meta Model Weights: [[2.70025424 0.47886961 1.02730886]]


In [27]:
# FINAL MODEL TESTING!!!

logreg_test_probs = logreg_l2.predict_proba(X_test)[:, 1]
rf_test_probs     = rf.predict_proba(X_test)[:, 1]
gb_test_probs     = gb.predict_proba(X_test)[:, 1]


X_meta_test = np.column_stack((
    logreg_test_probs,
    rf_test_probs,
    gb_test_probs
))


ensemble_probs = meta_model.predict_proba(X_meta_test)[:, 1]
ensemble_preds = (ensemble_probs >= 0.5).astype(int)


print("=== FINAL TEST SET RESULTS (20% HOLDOUT) ===")

print("Accuracy:", accuracy_score(y_test, ensemble_preds))
print("Log Loss:", log_loss(y_test, ensemble_probs))

print("\nClassification Report:")
print(classification_report(y_test, ensemble_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, ensemble_preds))



=== FINAL TEST SET RESULTS (20% HOLDOUT) ===
Accuracy: 0.6602596526723993
Log Loss: 0.6142246810924887

Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.43      0.51      2482
           1       0.67      0.83      0.74      3449

    accuracy                           0.66      5931
   macro avg       0.65      0.63      0.63      5931
weighted avg       0.66      0.66      0.64      5931


Confusion Matrix:
[[1058 1424]
 [ 591 2858]]

--- Sanity Checks ---
Probability range: 0.17375029525563365 to 0.8813911347697352
First 10 predictions: [1 0 0 0 0 1 1 1 1 1]


# Final Notes:

## Final Model Notes
The model is biased towards predicting home teams to win games shown in the recall section. 43% change it guesses correctly that a home team lost. 

## Changes to model/Things to note
Before the final results making I adjusted some code structure to ensure there was no data leakage. Then I adjusted the amount of rolling game to 10 instead of 5 this means that each rolling variable has more games of data to go off but more games at the beginning of the season are not included in the training. (ESPN analyst recommended it and it did in fact bring up the overall accuracy score by about 0.3%) 

Make sure to inlude a table in the results section that has all the major comapnies accuracy scores for NBA/ pro basketball. Found on "thepredictiontracker.com".
